# 🐔 Pipeline Pelatihan Ulang (Retraining) - Deteksi Penyakit Ayam
**Didesain khusus untuk Google Colab**

## 📖 Dokumentasi & Arsitektur
Sesuai dengan rancangan perbandingan pada skripsi Anda, notebook ini melatih dua jenis model secara berurutan:
1. **Baseline Custom CNN** (sebagai pembanding dasar)
2. **MobileNetV2 Feature Extraction** (Transfer Learning tahap awal)
3. **MobileNetV2 Fine-Tuning** (Adaptasi penuh pada tekstur data kandang)

Setiap tahap akan dievaluasi (*Confusion Matrix* & *Classification Report*) secara terpisah untuk bahan perbandingan di Bab 4.

In [ ]:
# ==========================================
# 1. PERSIAPAN ENVIRONMENT & SETUP
# ==========================================
from google.colab import files

import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import seaborn as sns

print(f"TensorFlow Version: {tf.__version__}")

In [ ]:
# ==========================================
# 2. EKSTRAK DATASET MERGED
# ==========================================
# Pastikan Anda sudah meng-upload archive_merged.zip secara manual ke Session Storage Colab (/content/)
ZIP_PATH = "/content/archive_merged.zip"
EXTRACT_PATH = "/content/dataset"

!unzip -q "{ZIP_PATH}" -d "{EXTRACT_PATH}"

In [ ]:
# ==========================================
# 3. PEMBACAAN CSV & SPLITTING DATASET
# ==========================================
IMG_DIR = f"{EXTRACT_PATH}/archive/Train"
CSV_PATH = f"{EXTRACT_PATH}/archive/train_data.csv"

df = pd.read_csv(CSV_PATH)
print("Total dataset:", len(df))

# Stratified Split: 70% Train, 15% Val, 15% Test
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(rotation_range=15, horizontal_flip=True)
test_datagen = ImageDataGenerator()

train_ds = train_datagen.flow_from_dataframe(
    dataframe=train_df, directory=IMG_DIR, x_col='images', y_col='label',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True
)

val_ds = test_datagen.flow_from_dataframe(
    dataframe=val_df, directory=IMG_DIR, x_col='images', y_col='label',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

test_ds = test_datagen.flow_from_dataframe(
    dataframe=test_df, directory=IMG_DIR, x_col='images', y_col='label',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

class_names = list(train_ds.class_indices.keys())
print("Class mapping:", train_ds.class_indices)

In [ ]:
# ==========================================
# 4. KALKULASI CLASS WEIGHTS & FUNGSI EVALUASI
# ==========================================
y_train = train_ds.classes
class_weights = compute_class_weight(
    class_weight="balanced", classes=np.unique(y_train), y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

print("\nBobot Kelas (Class Weights) - Penalti Ekstra untuk NCD:")
for i, weight in class_weight_dict.items():
    print(f"{class_names[i]}: {weight:.4f}")

# Fungsi bantu untuk menghasilkan Report & Confusion Matrix otomatis
def evaluate_and_plot(model, title, prefix):
    print(f"\n{'='*50}")
    print(f"EVALUASI TEST SET: {title}")
    print(f"{'='*50}")
    loss, acc = model.evaluate(test_ds, verbose=0)
    print(f"Akurasi Test: {acc:.4f} | Loss Test: {loss:.4f}\n")
    
    test_ds.reset()
    preds = model.predict(test_ds, verbose=0)
    y_pred = np.argmax(preds, axis=1)
    y_true = test_ds.classes
    
    print("CLASSIFICATION REPORT:")
    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    df_report = pd.DataFrame(report).transpose()
    df_report.to_csv(f"{prefix}_report.csv")
    print(classification_report(y_true, y_pred, target_names=class_names))
    
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, xticks_rotation=45)
    plt.title(f"Confusion Matrix - {title}")
    plt.tight_layout()
    plt.savefig(f"{prefix}_cm.png", dpi=300)
    plt.show()

---
# 🧪 MODEL 1: BASELINE CUSTOM CNN

In [ ]:
# ==========================================
# 5. TRAINING & EVALUASI BASELINE CNN
# ==========================================
baseline_inputs = tf.keras.Input(shape=(224, 224, 3))
x = layers.Rescaling(1./127.5, offset=-1)(baseline_inputs)
x = layers.Conv2D(32, (3, 3), activation='relu')(x)
x = layers.MaxPooling2D(2, 2)(x)
x = layers.Conv2D(64, (3, 3), activation='relu')(x)
x = layers.MaxPooling2D(2, 2)(x)
x = layers.Conv2D(128, (3, 3), activation='relu')(x)
x = layers.MaxPooling2D(2, 2)(x)
x = layers.Flatten()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
baseline_outputs = layers.Dense(len(class_names), activation='softmax')(x)

baseline_model = tf.keras.Model(baseline_inputs, baseline_outputs)
baseline_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("\n--- TRAINING BASELINE CNN ---")
history_baseline = baseline_model.fit(
    train_ds, validation_data=val_ds, epochs=10, class_weight=class_weight_dict
)

# Langsung panggil Evaluasi setelah selesai training baseline
evaluate_and_plot(baseline_model, "Baseline Custom CNN", "baseline")

---
# 🚀 MODEL 2: MOBILENETV2 (Transfer Learning)

In [ ]:
# ==========================================
# 6. ARSITEKTUR MOBILENETV2
# ==========================================
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights="imagenet"
)

base_model.trainable = False  # Freeze untuk Tahap 1

inputs = tf.keras.Input(shape=(224, 224, 3))
x = layers.Rescaling(1./127.5, offset=-1)(inputs) 
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

In [ ]:
# ==========================================
# 7. TAHAP 1 (FEATURE EXTRACTION) & EVALUASI
# ==========================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("\n--- TRAINING MOBILENETV2 (FEATURE EXTRACTION) ---")
history_fe = model.fit(
    train_ds, validation_data=val_ds, epochs=10, class_weight=class_weight_dict
)

# Evaluasi kondisi model SETELAH feature extraction tapi SEBELUM fine-tuning
evaluate_and_plot(model, "MobileNetV2 (Feature Extraction)", "mobilenet_fe")

In [ ]:
# ==========================================
# 8. TAHAP 2 (FINE-TUNING) & EVALUASI
# ==========================================
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_list = [
    callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(factor=0.2, patience=3)
]

print("\n--- TRAINING MOBILENETV2 (FINE-TUNING) ---")
history_ft = model.fit(
    train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks_list, class_weight=class_weight_dict
)

# Evaluasi akhir model SETELAH fine-tuning
evaluate_and_plot(model, "MobileNetV2 (Fine-Tuned)", "mobilenet_ft")

In [ ]:
# ==========================================
# 9. PEMBUATAN GRAFIK TRAINING
# ==========================================
def plot_history(hist, title, filename):
    acc = hist.history['accuracy']
    val_acc = hist.history['val_accuracy']
    loss = hist.history['loss']
    val_loss = hist.history['val_loss']
    epochs = range(1, len(acc) + 1)

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, label='Training Accuracy')
    plt.plot(epochs, val_acc, label='Validation Accuracy')
    plt.title(f'{title} - Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, label='Training Loss')
    plt.plot(epochs, val_loss, label='Validation Loss')
    plt.title(f'{title} - Loss')
    plt.legend()

    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.show()

plot_history(history_baseline, "Baseline Custom CNN", "grafik_baseline_cnn.png")
plot_history(history_fe, "MobileNetV2 Feature Extraction", "grafik_feature_extraction.png")
plot_history(history_ft, "MobileNetV2 Fine-Tuned", "grafik_fine_tuning.png")

In [ ]:
# ==========================================
# 10. SIMPAN MODEL & UNDUH SEMUA BERKAS
# ==========================================
MODEL_NAME = "mobilenetv2_finetuned_best.keras"
model.save(MODEL_NAME)
print(f"\nModel akhir (Fine-Tuned) berhasil disimpan lokal sebagai: {MODEL_NAME}")

print("\nMemulai proses download seluruh berkas hasil eksperimen...")

# Unduh Model
files.download(MODEL_NAME)

# Unduh Grafik Training (Accuracy & Loss)
files.download("grafik_baseline_cnn.png")
files.download("grafik_feature_extraction.png")
files.download("grafik_fine_tuning.png")

# Unduh Confusion Matrix (CM)
files.download("baseline_cm.png")
files.download("mobilenet_fe_cm.png")
files.download("mobilenet_ft_cm.png")

# Unduh Classification Report (CSV)
files.download("baseline_report.csv")
files.download("mobilenet_fe_report.csv")
files.download("mobilenet_ft_report.csv")

print("✅ 1 Model, 3 Grafik Training, 3 Confusion Matrix, dan 3 Classification Report telah dikirim ke peramban (browser) Anda!")